# EEG Classification: Alzheimer's Disease vs Healthy Controls (AD vs CN)

**Approach:** XGBoost on hand-crafted EEG features (Relative Band Power + Spectral Coherence Connectivity)

**Pipeline:**
1. Mount Google Drive and load data
2. Subject-level 3-way split: Train / Validation / Test (70 / 15 / 15)
3. Downsample 500 Hz → 128 Hz
4. Sliding window: 30s windows, 15s overlap
5. Feature extraction: RBP + SCC per epoch
6. XGBoost training with early stopping on validation set
7. Subject-level prediction aggregation
8. Benchmarks: Accuracy, Precision, Recall, F1
9. SHAP interpretability

## 0. Install Dependencies

In [ ]:
!pip install -q xgboost shap PyWavelets

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import pywt
import shap
import matplotlib.pyplot as plt
import os

from scipy.signal import welch, resample
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

np.random.seed(42)

## 3. Configuration

Update `DRIVE_BASE` to match where your data folder lives in Google Drive.

In [ ]:
DRIVE_BASE = '/content/drive/MyDrive'          # change if your folder is nested
DATA_DIR   = os.path.join(DRIVE_BASE, 'training')
LABEL_CSV  = os.path.join(DATA_DIR, 'train_label_mapping.csv')

ORIG_SFREQ = 500    # original sampling rate (Hz)
SFREQ      = 128    # target sampling rate after downsampling (Hz)
WIN_SEC    = 30     # sliding window length (seconds)
STEP_SEC   = 15     # step between windows — 15s overlap
WIN_SAMP   = WIN_SEC  * SFREQ
STEP_SAMP  = STEP_SEC * SFREQ

# Subject-level split ratios
TEST_SIZE  = 0.15   # 15% held out as test
VAL_SIZE   = 0.15   # 15% of remaining held out as validation

LABEL_MAP  = {'A': 1, 'C': 0}   # AD=1, CN=0

print(f"Window: {WIN_SEC}s | Step: {STEP_SEC}s | Samples/window: {WIN_SAMP}")

## 4. Helper Functions

In [ ]:
def load_npy(path):
    return np.load(path, allow_pickle=True)


def downsample(eeg, orig_fs, target_fs):
    """Resample EEG (channels, time) from orig_fs to target_fs."""
    target_points = int(eeg.shape[1] * target_fs / orig_fs)
    return resample(eeg, target_points, axis=1)


def extract_features(epoch, sfreq):
    """
    Extract RBP and SCC features from a single 30-second epoch.

    Args:
        epoch:  numpy array (19 channels, WIN_SAMP timepoints)
        sfreq:  sampling frequency in Hz (128)

    Returns:
        1D feature vector of length 190  (5 bands × 19 channels × 2 features)
    """
    n_channels = epoch.shape[0]
    bands = [(0.5, 4), (4, 8), (8, 13), (13, 25), (25, 45)]   # delta/theta/alpha/beta/gamma

    # --- Relative Band Power (RBP) ---
    freqs, psd = welch(epoch, fs=sfreq, nperseg=sfreq * 2, axis=1)  # (channels, freqs)
    total_power = psd.sum(axis=1, keepdims=True)
    total_power[total_power == 0] = 1e-10

    rbp = np.zeros((len(bands), n_channels))
    for b, (fmin, fmax) in enumerate(bands):
        idx = (freqs >= fmin) & (freqs <= fmax)
        rbp[b] = psd[:, idx].sum(axis=1) / total_power.flatten()

    # --- Spectral Coherence Connectivity (SCC) via CWT ---
    morlet_freqs = np.array([2, 6, 10, 18, 35])   # representative frequency per band
    wavelet      = 'cmor1.5-1.0'
    scales       = (pywt.central_frequency(wavelet) * sfreq) / morlet_freqs

    coeffs = np.zeros((n_channels, len(morlet_freqs), epoch.shape[1]), dtype=np.complex128)
    for ch in range(n_channels):
        cwt_out, _ = pywt.cwt(epoch[ch], scales, wavelet, sampling_period=1/sfreq)
        coeffs[ch] = cwt_out

    scc = np.zeros((len(morlet_freqs), n_channels))
    for b in range(len(morlet_freqs)):
        seg  = coeffs[:, b, :]              # (channels, time)
        csd  = seg @ seg.conj().T           # cross-spectral density matrix
        pv   = np.diag(csd).real
        denom = np.sqrt(np.outer(pv, pv))
        denom[denom == 0] = 1e-10
        scc[b] = np.abs(csd / denom).mean(axis=1)

    return np.concatenate([rbp.flatten(), scc.flatten()])   # (190,)


def build_feature_names():
    channels   = ['Fp1','Fp2','F7','F3','Fz','F4','F8','T3','C3','Cz',
                  'C4','T4','T5','P3','Pz','P4','T6','O1','O2']
    band_names = ['Delta','Theta','Alpha','Beta','Gamma']
    names = [f'RBP_{b}_{ch}' for b in band_names for ch in channels]
    names += [f'SCC_{b}_{ch}' for b in band_names for ch in channels]
    return names

FEATURE_NAMES = build_feature_names()
print(f'Features per epoch: {len(FEATURE_NAMES)}')

## 5. Load Data (AD + CN only)

In [ ]:
df_labels = pd.read_csv(LABEL_CSV)
df = df_labels[df_labels['label'].isin(['A', 'C'])].reset_index(drop=True)
print(f"Subjects — AD: {(df['label']=='A').sum()}, CN: {(df['label']=='C').sum()}, Total: {len(df)}")

subject_ids, raw_data, labels = [], [], []

for _, row in df.iterrows():
    sid    = row['anonymized_id']
    label  = row['label']
    folder = 'AD' if label == 'A' else 'CN'
    path   = os.path.join(DATA_DIR, folder, f"{sid}.npy")

    if not os.path.exists(path):
        print(f"  WARNING: missing {path}")
        continue

    eeg = load_npy(path)                        # (19, time @ 500 Hz)
    eeg = downsample(eeg, ORIG_SFREQ, SFREQ)   # (19, time @ 128 Hz)

    subject_ids.append(sid)
    raw_data.append(eeg)
    labels.append(LABEL_MAP[label])

subject_ids = np.array(subject_ids)
labels      = np.array(labels)
print(f"Successfully loaded {len(subject_ids)} subjects")

## 6. Subject-Level 3-Way Split: Train / Validation / Test

**All epochs from a subject stay in the same split — no data leakage.**

Split strategy:
- Step 1: hold out 15% of subjects as **Test**
- Step 2: from the remaining 85%, hold out ~18% as **Validation** (≈15% of total)
- Remaining subjects → **Train**

In [ ]:
# Step 1: split off test subjects
gss1 = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=42)
trainval_idx, test_idx = next(gss1.split(subject_ids, labels, groups=subject_ids))

# Step 2: split remaining into train + validation
# val_size relative to trainval pool so that it equals ~15% of total
val_size_relative = VAL_SIZE / (1 - TEST_SIZE)
gss2 = GroupShuffleSplit(n_splits=1, test_size=val_size_relative, random_state=42)
sub_trainval = subject_ids[trainval_idx]
lab_trainval = labels[trainval_idx]
train_sub_idx, val_sub_idx = next(gss2.split(sub_trainval, lab_trainval, groups=sub_trainval))

# Map back to global indices
train_idx = trainval_idx[train_sub_idx]
val_idx   = trainval_idx[val_sub_idx]

def subset(idx):
    return subject_ids[idx], [raw_data[i] for i in idx], labels[idx]

train_sids, train_data, train_labels = subset(train_idx)
val_sids,   val_data,   val_labels   = subset(val_idx)
test_sids,  test_data,  test_labels  = subset(test_idx)

print(f"Train : {len(train_sids)} subjects  (AD={train_labels.sum()}, CN={(train_labels==0).sum()})")
print(f"Val   : {len(val_sids)} subjects  (AD={val_labels.sum()}, CN={(val_labels==0).sum()})")
print(f"Test  : {len(test_sids)} subjects  (AD={test_labels.sum()}, CN={(test_labels==0).sum()})")

## 7. Feature Extraction — Sliding Window

30s window, 15s step → 50% overlap. Each window → 190 features.

In [ ]:
def extract_all_epochs(data_list, sids, sid_labels, split_name):
    """Returns X (n_epochs, 190), y (n_epochs,), groups (n_epochs,)."""
    X_list, y_list, g_list = [], [], []
    print(f"\nExtracting {split_name} features...")

    for eeg, sid, label in zip(data_list, sids, sid_labels):
        n_points    = eeg.shape[1]
        epoch_count = 0

        for start in range(0, n_points - WIN_SAMP + 1, STEP_SAMP):
            epoch = eeg[:, start : start + WIN_SAMP]
            X_list.append(extract_features(epoch, SFREQ))
            y_list.append(label)
            g_list.append(sid)
            epoch_count += 1

        print(f"  Subject {sid}: {epoch_count} epochs")

    X = np.array(X_list)
    y = np.array(y_list)
    g = np.array(g_list)
    print(f"  → {X.shape[0]} total epochs, {X.shape[1]} features each")
    return X, y, g


X_train, y_train, g_train = extract_all_epochs(train_data, train_sids, train_labels, 'Train')
X_val,   y_val,   g_val   = extract_all_epochs(val_data,   val_sids,   val_labels,   'Validation')
X_test,  y_test,  g_test  = extract_all_epochs(test_data,  test_sids,  test_labels,  'Test')

## 8. Feature Scaling

Fit scaler on train only — apply same transform to val and test.

In [ ]:
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)
print("Scaling done.")

## 9. Train XGBoost with Early Stopping on Validation Set

The validation set controls early stopping — training halts when val loss stops improving.

In [ ]:
model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

print(f"\nBest iteration: {model.best_iteration}")

## 10. Subject-Level Prediction Aggregation

Average epoch-level probabilities per subject → threshold at 0.5.

In [ ]:
def aggregate_predictions(model, X, groups, epoch_labels):
    """
    Aggregates epoch-level P(AD) to one prediction per subject.

    Returns: sids, y_true, y_pred, y_prob
    """
    probs = model.predict_proba(X)[:, 1]
    sids, y_true, y_pred, y_prob = [], [], [], []

    for sid in np.unique(groups):
        mask     = groups == sid
        avg_prob = probs[mask].mean()
        sids.append(sid)
        y_true.append(epoch_labels[mask][0])
        y_pred.append(int(avg_prob >= 0.5))
        y_prob.append(avg_prob)

    return np.array(sids), np.array(y_true), np.array(y_pred), np.array(y_prob)

## 11. Benchmarks — Accuracy, Precision, Recall, F1

In [ ]:
def print_benchmarks(split_name, y_true, y_pred):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    print(f"\n{'='*40}")
    print(f" {split_name} Results (subject-level)")
    print(f"{'='*40}")
    print(f"  Accuracy  : {acc:.3f}")
    print(f"  Precision : {prec:.3f}")
    print(f"  Recall    : {rec:.3f}")
    print(f"  F1 Score  : {f1:.3f}")
    print(f"{'='*40}")
    return acc, prec, rec, f1


# Validation benchmarks
_, val_true, val_pred, _ = aggregate_predictions(model, X_val, g_val, y_val)
print_benchmarks('Validation', val_true, val_pred)

# Test benchmarks
test_sids_out, test_true, test_pred, test_prob = aggregate_predictions(model, X_test, g_test, y_test)
print_benchmarks('Test', test_true, test_pred)

## 12. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, y_t, y_p, title in [
    (axes[0], val_true,  val_pred,  'Validation'),
    (axes[1], test_true, test_pred, 'Test')
]:
    cm   = confusion_matrix(y_t, y_p)
    disp = ConfusionMatrixDisplay(cm, display_labels=['CN', 'AD'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{title} — Subject-Level')

plt.tight_layout()
plt.show()

## 13. Training Loss Curve

In [ ]:
val_logloss = model.evals_result()['validation_0']['logloss']

plt.figure(figsize=(8, 4))
plt.plot(val_logloss, label='Validation Log-Loss', color='steelblue')
plt.axvline(model.best_iteration, color='tomato', linestyle='--', label=f'Best iteration ({model.best_iteration})')
plt.xlabel('Boosting Round')
plt.ylabel('Log-Loss')
plt.title('XGBoost Validation Loss Curve')
plt.legend()
plt.tight_layout()
plt.show()

## 14. Band Power Visualization (EDA)

Compare mean relative band power between AD and CN subjects.

In [ ]:
band_names = ['Delta', 'Theta', 'Alpha', 'Beta', 'Gamma']
n_bands, n_ch = 5, 19

rbp_all = X_train[:, :n_bands * n_ch].reshape(-1, n_bands, n_ch)
rbp_ad  = rbp_all[y_train == 1].mean(axis=(0, 2))
rbp_cn  = rbp_all[y_train == 0].mean(axis=(0, 2))

x, w = np.arange(n_bands), 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, rbp_ad, w, label='AD', color='tomato')
ax.bar(x + w/2, rbp_cn, w, label='CN', color='steelblue')
ax.set_xticks(x)
ax.set_xticklabels(band_names)
ax.set_ylabel('Mean Relative Band Power (standardised)')
ax.set_title('Average RBP: AD vs CN (Training Set)')
ax.legend()
plt.tight_layout()
plt.show()

## 15. SHAP Interpretability

Which EEG features drive the AD classification?

In [ ]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Beeswarm — shows direction and magnitude
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_NAMES, max_display=20)

In [ ]:
# Bar chart — top 20 features by mean |SHAP|
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_NAMES, plot_type='bar', max_display=20)

## 16. Inference on Released Test Data (Sunday)

Update `TEST_DATA_DIR` to point to the released test folder in Drive.

In [ ]:
TEST_DATA_DIR = os.path.join(DRIVE_BASE, 'test')   # update path when test data is released

test_files = [f for f in os.listdir(TEST_DATA_DIR) if f.endswith('.npy')]
inf_sids, inf_eegs = [], []

for fname in test_files:
    sid = int(fname.replace('.npy', ''))
    eeg = load_npy(os.path.join(TEST_DATA_DIR, fname))
    eeg = downsample(eeg, ORIG_SFREQ, SFREQ)
    inf_sids.append(sid)
    inf_eegs.append(eeg)

# Feature extraction
X_inf_list, g_inf_list = [], []
for eeg, sid in zip(inf_eegs, inf_sids):
    for start in range(0, eeg.shape[1] - WIN_SAMP + 1, STEP_SAMP):
        X_inf_list.append(extract_features(eeg[:, start:start + WIN_SAMP], SFREQ))
        g_inf_list.append(sid)

X_inf = scaler.transform(np.array(X_inf_list))
g_inf = np.array(g_inf_list)

# Subject-level predictions
probs_inf = model.predict_proba(X_inf)[:, 1]
results   = []
for sid in np.unique(g_inf):
    mask     = g_inf == sid
    avg_prob = probs_inf[mask].mean()
    results.append({
        'anonymized_id': sid,
        'label': 'A' if avg_prob >= 0.5 else 'C',
        'prob_AD': round(avg_prob, 4)
    })

df_out = pd.DataFrame(results).sort_values('anonymized_id')
df_out.to_csv('/content/drive/MyDrive/predictions.csv', index=False)
print(df_out.to_string(index=False))
print("\nSaved to predictions.csv")